In [0]:
%run ../config/config

In [0]:
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")

from pyspark.sql import functions as F
from pyspark.sql.types import *

raw_base = "/Volumes/dev/bronze/bronze_volume/raw/"
checkpoint_base = "/Volumes/dev/bronze/bronze_volume/checkpoints/"
schema_base = "/Volumes/dev/bronze/bronze_volume/schema/"


# Define Bronze Schemas
customer_schema = StructType([
    StructField('user_id', IntegerType(), True),
    StructField('user_name', StringType(), True),
    StructField('name_surname', StringType(), True),
    StructField('status', IntegerType(), True),
    StructField('user_gender', StringType(), True),
    StructField('user_birth_date', DateType(), True),
    StructField('region', StringType(), True),
    StructField('city', StringType(), True),
    StructField('town', StringType(), True),
    StructField('district', StringType(), True),
    StructField('address_text', StringType(), True),
    StructField('last_modified_date', TimestampType(), True)
])

branch_schema = StructType([
    StructField('branch_id', StringType(), True),
    StructField('region', StringType(), True),
    StructField('city', StringType(), True),
    StructField('town', StringType(), True),
    StructField('branch_town', StringType(), True),
    StructField('lat', LongType(), True),
    StructField('lon', LongType(), True)
])

category_schema = StructType([
    StructField('item_id', IntegerType(), True),
    StructField('category1', StringType(), True),
    StructField('category1_id', StringType(), True),
    StructField('category2', StringType(), True),
    StructField('category2_id', StringType(), True),
    StructField('category3', StringType(), True),
    StructField('category3_id', StringType(), True),
    StructField('category4', StringType(), True),
    StructField('category4_id', StringType(), True),
    StructField('brand', StringType(), True),
    StructField('item_code', IntegerType(), True),
    StructField('item_name', StringType(), True)
])

order_schema = StructType([
    StructField('order_id', IntegerType(), True),
    StructField('branch_id', StringType(), True),
    StructField('date', TimestampType(), True),
    StructField('user_id', IntegerType(), True),
    StructField('name_surname', StringType(), True),
    StructField('total_basket', StringType(), True)
])

order_detail_schema = StructType([
    StructField('order_id', IntegerType(), True),
    StructField('order_detail_id', IntegerType(), True),
    StructField('amount', IntegerType(), True),
    StructField('unit_price', StringType(), True),
    StructField('total_price', StringType(), True),
    StructField('item_id', IntegerType(), True),
    StructField('item_code', StringType(), True)
])


# Ingestion Configuration
INGESTION_CONFIG = {
    "customer": {
        "schema": customer_schema,
        "source_path": f"{raw_base}customers/",
        "target_table": f"{bronze_catalog}.customers"
    },

    "branch": {
        "schema": branch_schema,
        "source_path": f"{raw_base}branches/",
        "target_table": f"{bronze_catalog}.branches"
    },

    "category": {
        "schema": category_schema,
        "source_path": f"{raw_base}categories/",
        "target_table": f"{bronze_catalog}.categories"
    },

    "order": {
        "schema": order_schema,
        "source_path": f"{raw_base}orders/",
        "target_table": f"{bronze_catalog}.orders"
    },

    "order_detail": {
        "schema": order_detail_schema,
        "source_path": f"{raw_base}order_details/",
        "target_table": f"{bronze_catalog}.order_details"
    }
}


# Ingestion Loop
for table, config in INGESTION_CONFIG.items():

    print(f"Starting ingestion for {table}")

    # Read raw data from bronze volume
    raw_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", f"{schema_base}{table}")
        .schema(config["schema"])
        .load(config["source_path"])
    )

    # Add metadata
    bronze_df = (
        raw_df
        .withColumn("_ingest_time", F.current_timestamp())
        .withColumn("_source_file", F.input_file_name())
    )

    # Write data to bronze delta table
    query = (
        bronze_df.writeStream
        .format("delta")
        .option("checkpointLocation", f"{checkpoint_base}{table}")
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(config["target_table"])
    )

    query.awaitTermination()


    # Row Count Validation
    record_count = spark.table(config["target_table"]).count()

    if record_count == 0:
        raise ValueError("Row count validation failed: No records ingested")

    print(f"Ingestion successful for {table} | rows: {record_count}")